In [1]:
from api_client import ForecastAPIClient
import sys
import os

# Récupère le répertoire du notebook courant
notebook_dir = os.getcwd()

# Monte d'un cran si nécessaire
parent_dir = os.path.dirname(notebook_dir)

# Ajoute au sys.path
sys.path.append(parent_dir)

from pipeline_prevision.utils.main_utils.utils import concat_all_data
import pickle

In [9]:
preprocessor = pickle.load(open(r"D:\Nouveau dossier\TitreRNCP_Bloc1\projet_prevision_energies_bloc5\final_models\preprocessor.pkl", "rb"))
a =preprocessor.feature_names_in_

In [10]:
a[1:5]

array(['SOLAR', 'BIOMASS', 'WIND_ONSHORE', 'NUCLEAR'], dtype=object)

In [18]:
api_client = ForecastAPIClient()
data = concat_all_data("2025-01-01", "2025-03-30")


In [19]:
data.head()

,temp,index,SOLAR,BIOMASS,WIND_ONSHORE,NUCLEAR,consommation_totale
timestamp,,,,,,,
2024-12-31 23:00:00,5.0,0,284,352,9687,46092,63773.00
2025-01-01 00:00:00,5.0,1,0,360,10063,44640,61601.50
2025-01-01 01:00:00,4.7,2,0,360,10213,44255,60828.00
2025-01-01 02:00:00,4.4,3,0,359,10498,42740,58473.75
2025-01-01 03:00:00,4.0,4,0,359,10740,41265,56430.25


In [6]:
y_pred, y_test, mae, mse = api_client.predict_multistep(data, 1)

Appel API vers: https://energiesforecasts-7f55a2300a2c.herokuapp.com/predict_multistep
Payload shape: 36 x 6
Réponse API - y_pred shape: (1, 6)
MAE: None, MSE: None


In [35]:
testX = data[-168:-24]
testY = data[-24:]

In [27]:
# datafirt
import pandas as pd
from scipy.stats import ks_2samp
def detect_dataset_drift( 
                              base_df: pd.DataFrame, 
                              current_df: pd.DataFrame, 
                              threshold: float = 0.05) -> bool:
                                  
    status = True
    report = {}                      
    for column in base_df.columns:
        is_same_distribution = ks_2samp(base_df[column], current_df[column])
        if threshold <= is_same_distribution.pvalue:
            is_found = False
        else:
            is_found = True
            status = False
        
        report.update(
                    {column: {
                        "pvalue": float(is_same_distribution.pvalue),
                        "drift_status": is_found
                    }}
                )
    return report


In [36]:
report = detect_dataset_drift(testX, testY)
report

{'temp': {'pvalue': 0.008989866690054988, 'drift_status': True},
 'index': {'pvalue': 2.7264985577140665e-29, 'drift_status': True},
 'SOLAR': {'pvalue': 0.5788689370362632, 'drift_status': False},
 'BIOMASS': {'pvalue': 7.178497172158731e-23, 'drift_status': True},
 'WIND_ONSHORE': {'pvalue': 0.024785412261336712, 'drift_status': True},
 'NUCLEAR': {'pvalue': 0.2687608824550046, 'drift_status': False},
 'consommation_totale': {'pvalue': 0.1163851077752821, 'drift_status': False}}